# Volume 9 Drills — Multi-Agent RL

Work through each drill problem. Code cells are provided where applicable.

## Drill 1 — MARL: Key Challenge vs Single-Agent RL

**Question:** What is the **key challenge** in multi-agent RL (MARL) that does not exist in single-agent RL?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The fundamental MARL challenge is **non-stationarity**:

In single-agent RL, the environment (MDP) is stationary — the transition and reward functions don't change. The agent can converge to an optimal policy.

In MARL, each agent observes a world that is **affected by all other agents' policies**. As other agents update their policies, the environment appears to change from any single agent's perspective. This breaks the Markov property and standard RL convergence guarantees.

**Additional challenges:**
- **Credit assignment:** In cooperative settings, which agent contributed to team reward?
- **Scalability:** The joint action space grows exponentially with agents.
- **Partial observability:** Agents typically have limited local views.
- **Equilibrium selection:** Multiple Nash equilibria may exist in competitive settings.
</details>

## Drill 2 — Nash Equilibrium: Definition

**Question:** Define **Nash Equilibrium** for a 2-player game.

**Answer (fill in):** ___

<details><summary>Answer</summary>

A **Nash Equilibrium** is a pair of strategies (π₁*, π₂*) such that **no player can improve their expected payoff by unilaterally changing their strategy**, given the other player's strategy is fixed:

$$\forall \pi_1:\ U_1(\pi_1^*, \pi_2^*) \geq U_1(\pi_1, \pi_2^*)$$
$$\forall \pi_2:\ U_2(\pi_1^*, \pi_2^*) \geq U_2(\pi_1^*, \pi_2)$$

**Key properties:**
- Every finite game has at least one Nash equilibrium (possibly in **mixed strategies**).
- NE is a **self-consistent** fixed point, not necessarily the socially optimal outcome.
- In zero-sum games (rock-paper-scissors), the Nash equilibrium is unique in mixed strategies.

**Example:** In the Prisoner's Dilemma, both defecting is the unique NE even though both cooperating gives higher joint reward.
</details>

## Drill 3 — IQL: Independence Assumption

**Question:** What is the **independence assumption** in Independent Q-Learning (IQL)?

**Answer (fill in):** ___

<details><summary>Answer</summary>

IQL applies **standard single-agent Q-learning independently to each agent**, treating all other agents as part of the environment.

**Independence assumption:** Each agent i learns Q_i(o_i, a_i) — a local Q-function that depends only on its own observation o_i and action a_i — while **ignoring** the actions and observations of other agents.

**Consequence:** From agent i's perspective, the environment appears **non-stationary** (because other agents are updating). This violates the stationarity assumption Q-learning requires for convergence.

**Despite this:** IQL often works surprisingly well in practice, especially in cooperative tasks. It's simple, scalable, and a useful baseline.

**Contrast with:** CTDE methods (QMIX, MADDPG) which use **centralized information during training** to handle non-stationarity.
</details>

## Drill 4 — CTDE: Why Centralized Training, Decentralized Execution?

**Question:** Why do many MARL algorithms use the **CTDE** (Centralized Training, Decentralized Execution) paradigm?

**Answer (fill in):** ___

<details><summary>Answer</summary>

**CTDE separates training from deployment:**

**Centralized training:**
- During training, agents can share observations, actions, and rewards with a **central critic** or coordinator.
- This provides richer information for credit assignment and resolves non-stationarity.
- Agents learn coordinated behaviors.

**Decentralized execution:**
- At deployment, each agent acts using only its **local observation**.
- No communication or central controller needed at runtime.
- Practical for real-world scenarios: communication may be unavailable, introduce latency, or be subject to failure.

**Why this works:** The centralized training provides better training signal; the local policy learns to act well based only on local information because it was trained with global context.

*Examples: QMIX, MADDPG, COMA, MAPPO.*
</details>

## Drill 5 — QMIX: Monotonicity Constraint

**Question:** What constraint does QMIX impose on the joint Q-function Q_tot? Why?

**Answer (fill in):** ___

<details><summary>Answer</summary>

QMIX imposes the **monotonicity constraint** (Individual-Global Maximum, IGM):

$$\frac{\partial Q_{tot}}{\partial Q_i} \geq 0 \quad \forall i$$

Q_tot is a monotonic function of each agent's Q_i values. This is enforced by using **non-negative weights** in the mixing network (weights come from a hypernetwork conditioned on global state, but are passed through an absolute value or softplus).

**Why this matters (IGM condition):**
$$\arg\max_{\mathbf{a}} Q_{tot}(\mathbf{s}, \mathbf{a}) = \left( \arg\max_{a_1} Q_1(o_1, a_1),\ \ldots,\ \arg\max_{a_n} Q_n(o_n, a_n) \right)$$

If monotonicity holds, the **joint greedy action** equals the combination of each agent's individually greedy actions. This means agents can act greedily and independently at execution time (decentralized) while the joint action is globally consistent.
</details>

## Drill 6 — MADDPG: Critic During Training

**Question:** How does the centralized critic in MADDPG help during training?

**Answer (fill in):** ___

<details><summary>Answer</summary>

MADDPG extends DDPG to multi-agent settings using CTDE:

- **Actors:** Each agent i has a local actor μ_i(o_i; θ_i) that observes only o_i (decentralized).
- **Critics:** Each agent i has a centralized critic Q_i(**o**, **a**; φ_i) that takes **all** agents' observations and actions as input.

**Benefits of the centralized critic:**

1. **Resolves non-stationarity:** Q_i can see all agents' policies changing, so the environment appears stationary to the critic.

2. **Better credit assignment:** Q_i can distinguish between "agent i's action was good" and "another agent's action caused the reward."

3. **Off-policy learning:** With access to all agents' actions, standard off-policy Q-learning works correctly.

The critic is only used during training; at execution, each agent uses only its local actor.
</details>

## Drill 7 — Code: Rock-Paper-Scissors and Nash Equilibrium

In [ ]:
import numpy as np

# Rock-Paper-Scissors payoff matrix for player 1
# Actions: 0=Rock, 1=Paper, 2=Scissors
# Payoff: +1 win, -1 lose, 0 draw
payoff = np.array([
    # R    P    S   (player 2)
    [ 0,  -1,   1],  # Rock    (player 1)
    [ 1,   0,  -1],  # Paper
    [-1,   1,   0],  # Scissors
])

def expected_payoff(p1_strategy, p2_strategy):
    """Expected payoff for player 1 given mixed strategies."""
    return p1_strategy @ payoff @ p2_strategy

# Nash Equilibrium of RPS: uniform mixed strategy [1/3, 1/3, 1/3]
nash = np.array([1/3, 1/3, 1/3])

print("Nash Equilibrium: each action with probability 1/3")
print(f"Expected payoff at Nash: {expected_payoff(nash, nash):.4f} (should be 0)")
print()

# Show that no player can benefit by deviating
strategies = {
    "Always Rock": np.array([1.0, 0.0, 0.0]),
    "Always Paper": np.array([0.0, 1.0, 0.0]),
    "Always Scissors": np.array([0.0, 0.0, 1.0]),
    "Nash [1/3,1/3,1/3]": nash,
}

print("If player 2 plays Nash, player 1's expected payoff with different strategies:")
for name, s in strategies.items():
    ep = expected_payoff(s, nash)
    print(f"  {name}: {ep:.4f}")

print()
print("All strategies yield 0 against Nash → no incentive to deviate → it's a Nash Equilibrium!")

## Drill 8 — Communication: Benefits for Agents

**Question:** Why might agents benefit from **sharing observations** with each other?

**Answer (fill in):** ___

<details><summary>Answer</summary>

In partially observable environments, each agent has a limited local view. Communication addresses this in several ways:

1. **Filling observational gaps:** Agent A may observe something relevant to Agent B's decision. By sharing, B can make a better-informed choice.
   *Example: In StarCraft, a scout can relay enemy positions to allies that can't see them.*

2. **Coordinating actions:** Agents can agree on a joint strategy ("I'll go left, you go right") to avoid redundancy or conflict.

3. **Implicit state sharing:** Sharing observations effectively converts a partially observable problem toward a fully observable one.

**Challenges of communication:**
- Bandwidth constraints (limited messages per step)
- Deciding **what** to communicate (learned communication: CommNet, DIAL, QMIX+comm)
- Adversarial settings: agents may lie or mislead

**Emergent communication:** Agents can also learn to develop communication protocols from scratch (e.g., the famous OpenAI emergent tool-use experiments).
</details>

## Drill 9 — Self-Play: Generating Curriculum

**Question:** How does **self-play** generate a natural curriculum for training agents?

**Answer (fill in):** ___

<details><summary>Answer</summary>

In self-play, an agent trains by playing **against copies of itself** (or past versions of itself):

1. **Automatic curriculum:** As the agent improves, its opponents also improve (since opponents are copies). The agent always faces an **appropriately challenging** opponent — neither too easy nor too hard.

2. **Arms race dynamic:** Each improvement in one player is immediately matched by the opponent, producing continuous pressure to improve.

3. **Diversity via historical opponents:** Training against a **pool of past policies** (rather than only the latest) prevents cyclic behavior (rock-paper-scissors loops) and ensures robust strategies.

**Examples:**
- AlphaGo/AlphaZero: trained via self-play from scratch, surpassed all human players.
- OpenAI Five: Dota 2 at superhuman level.
- OpenAI Hide-and-Seek: emergent tool use and strategy from self-play.

Self-play is particularly powerful in **zero-sum** games but also works in cooperative tasks.
</details>

## Drill 10 — Emergent Behavior: MARL Literature Example

**Question:** Give **one concrete example** of emergent behavior from MARL research.

**Answer (fill in):** ___

<details><summary>Answer</summary>

**OpenAI Hide-and-Seek (Baker et al., 2019)**

Two teams (hiders and seekers) compete in a physics simulation. Neither is given any strategy — they only receive +1/−1 based on whether hiders are visible.

Through self-play, the following behaviors **emerged** in sequence:

1. **Random movement** → seekers run toward hiders.
2. Hiders learn to **run away and hide** behind obstacles.
3. Seekers learn to **move ramps** to jump over walls.
4. Hiders learn to **move ramps inside** so seekers can't use them.
5. Seekers learn to **surf boxes** (unintended physics exploit) to enter rooms.
6. Hiders learn to **lock all objects** before seekers can grab them.

None of these strategies were programmed — they arose from pure reward-driven self-play, demonstrating **emergent multi-step tool use** and strategic planning.

*Other examples: AlphaZero discovering novel chess openings, emergent language in referential games.*
</details>